# Accessing Spatial Data

<div style="display: flex; justify-content: center;">
    <img src="https://bg.copernicus.org/articles/20/4551/2023/bg-20-4551-2023-avatar-web.png" alt="Credit: Copernicus seagrass mapping from sateillite data example image" style="width: 400px;"/>
    <img src="https://floridakeys.noaa.gov/review/graphics/seagrasswg.jpg" alt="Credit: Seagrass Beds. Florida Keys National Marine Sanctuary." style="width: 400px;"/>
</div>


---

## Overview
The purpose of this notebook is to loop through `sentinel2_data` year and `.SAFE` files containing satellite imagery, which will output a list of `band_data` containing the extracted spectral band information to be loaded as a raster grid and used in further notebooks.

1. Sentinel-2 Data Structure
2. Loading Sentinel-2 (S2) Files
3. Extracting band file paths.
4. Saving `band_data` as `.pkl` File

Sentinel-2 satelliete data was collected via [Copernicus Data Space Ecosystem](https://dataspace.copernicus.eu/data-collections/copernicus-sentinel-missions/sentinel-2). Downloaded for the years 2015-2024, around the AOI: Florida Bay and/or Upper Keys Florida.


## Sentinel-2 Data Structure
Sentinel-2 Data is stored in raster format with x and y coordinates. The spectral bands that will be used for indice calculations are within a `.SAFE` folder, and stored as `.jp2` file within a `granule` folder inside of `.SAFE`.

To make it easier for Python to loop through, files were previously extracted into corresponding `year` folders within the `sentinel2_data` folder. The structure of the Sentinel-2 data files is as follows, organized by year folders from 2015 to 2024.

sentinel2_data
* Year
     * S2A... .SAFE/
         * Granule/
             *   IMG_DATA/
                 * B02.jp2
                 * B03.jp2
                 * B04.jp2
                 * B08.jp2
### Modules/Functions to be demonstrated:
* `os` - (built-in) allows user to directly interact with system directories and file paths.
* `glob` - built in module that uses string pattern matching to find path names with `os` in the system directory.
* `pickle` - (built-in) converts data into storable format for reuse.


---

In [2]:
!pip install rioxarray

   ---------------------------------------- 0.0/6.3 MB ? eta -:--:--
   ---------------------------------------- 6.3/6.3 MB 35.4 MB/s  0:00:00

   ---------------------------------------- 0/2 [pyproj]
   ---------------------------------------- 0/2 [pyproj]
   -------------------- ------------------- 1/2 [rioxarray]
   ---------------------------------------- 2/2 [rioxarray]



### Imports

In [3]:
import sys
import os
import glob
import xarray as xr
import rioxarray as rxr
import rasterio
import pickle

## Loading S2 Files

The first step is to set the path to the folder containing the `sentinel2_data` by defining the directory and sorting by year to match the structure of the S2 folder. 

In [4]:
data_dir = "C:/Users/sulli/DATA_Science/seagrass_project/sentinel2_data"
years = sorted(os.listdir(data_dir))

print("Years:", years)

Years: ['2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024']


## Extracting the Band File Path
Using `for` loops and the Python `glob` module and the function `os.path.join()`to loop through years files to locate the number of `SAFE` files, or number of scenes per year. Scenes are snapshots of Earth taking over time. Another `for` loop will loop through the `.SAFE` files to combine the directories of the spectral band `.jp2` files, and extract to `band_data`. The length of `band_data` is the number of valid scenes within the dataset.

| File Name | Spectral Band Color |
|---|---|
| BO4 | Red |
| B02 | Blue |
| B03 | Green |
| B08 | NIR (near infrared) |

In [5]:
band_data = []

for year in years:
    year_path = os.path.join(data_dir, year)

    # find all safe files
    safe_folders = glob.glob(os.path.join(year_path, "*.SAFE"))
    
    print(year, "SAFE folders:", len(safe_folders))

    for safe in safe_folders:
        # search inside SAFE for jp2 files
        jp2_files = glob.glob(os.path.join(safe, "**/*.jp2"), recursive=True)

        #identify bands
        bands_path = {
            "blue": [f for f in jp2_files if "_B02_10m" in f],
            "green": [f for f in jp2_files if "_B03_10m" in f],
            "red": [f for f in jp2_files if "_B04_10m" in f],
            "nir": [f for f in jp2_files if "_B08_10m" in f],
        }

    # store results in band_data list dictionary
        band_data.append({
            "year": year,
            "safe_folder": safe,
            "bands": bands_path
        })

print("Total valid Scenes:", len(band_data))

2015 SAFE folders: 1
2016 SAFE folders: 5
2017 SAFE folders: 6
2018 SAFE folders: 7
2019 SAFE folders: 5
2020 SAFE folders: 2
2021 SAFE folders: 16
2022 SAFE folders: 9
2023 SAFE folders: 22
2024 SAFE folders: 17
Total valid Scenes: 90


In [6]:
type(band_data)

list

## Convert and Save `band_data`
### **Memory efficiency**
`jp2` files are very large and may cause memory issues during clipping and masking, therefore we need to convert the bands to smaller and easier to manage `.tif` files for the reproducibility of the workflow. (Warning: Slow output)

In [13]:
###CHATGPT
# Output folder for TIFs
output_folder = "band_data_tifs"
os.makedirs(output_folder, exist_ok=True)

# New list to store TIF paths
band_data_tifs = []

# Sort band_data by year first
band_data_sorted = sorted(band_data, key=lambda x: x['year'])

# Loop through each scene
for scene in band_data_sorted:
    # Loop over all bands in this scene
    for band_name, jp2_list in scene['bands'].items():
        for jp2_file in jp2_list:
            # Create TIF filename
            filename = os.path.basename(jp2_file).replace(".jp2", ".tif")
            out_path = os.path.join(output_folder, filename)

            # Convert JP2 → TIF
            with rasterio.open(jp2_file) as src:
                profile = src.profile
                profile.update(driver='GTiff', compress='lzw')

                with rasterio.open(out_path, 'w', **profile) as dst:
                    dst.write(src.read())

            # Add TIF path to list
            band_data_tifs.append(out_path)
            print(f"Converted: {filename} → Year {scene['year']}")

# band_data_tifs is now a flat list of all converted TIFs, sorted by year
print(f"Total TIFs converted: {len(band_data_tifs)}")

Converted: T17RNH_20151004T160016_B02_10m.tif → Year 2015
Converted: T17RNH_20151004T160016_B03_10m.tif → Year 2015
Converted: T17RNH_20151004T160016_B04_10m.tif → Year 2015
Converted: T17RNH_20151004T160016_B08_10m.tif → Year 2015
Converted: T17RNH_20160501T155412_B02_10m.tif → Year 2016
Converted: T17RNH_20160501T155412_B03_10m.tif → Year 2016
Converted: T17RNH_20160501T155412_B04_10m.tif → Year 2016
Converted: T17RNH_20160501T155412_B08_10m.tif → Year 2016
Converted: T17RNH_20160521T160522_B02_10m.tif → Year 2016
Converted: T17RNH_20160521T160522_B03_10m.tif → Year 2016
Converted: T17RNH_20160521T160522_B04_10m.tif → Year 2016
Converted: T17RNH_20160521T160522_B08_10m.tif → Year 2016
Converted: T17RNH_20160630T160512_B02_10m.tif → Year 2016
Converted: T17RNH_20160630T160512_B03_10m.tif → Year 2016
Converted: T17RNH_20160630T160512_B04_10m.tif → Year 2016
Converted: T17RNH_20160630T160512_B08_10m.tif → Year 2016
Converted: T17RNH_20160720T160512_B02_10m.tif → Year 2016
Converted: T17

In [ ]:
type(band_dataa_tifs)

#### Saving `band_data_tifs` list as a Pickle File

In [8]:
output_path = "C:/Users/sulli/DATA_Science/seagrass_project/processed/band_data.pkl"

#make sure folder exists
os.makedirs(os.path.dirname(output_path), exist_ok=True)

with open(output_path, "wb") as f:
    pickle.dump(band_data_tifs, f)

print("Saved band_data_tifs as band_data.")

Saved band_data.


---

## Summary
The output of this notebook is a saved Python pickle file containing the raster `band_data` that can be uploaded into the next notebook. **Converting `jp2` to `tif` is an important step** as it will help make for clean preprocessing.

### Preprocessing Spatial Data
The following notebook will open `band_data` for preprocess by creating individual raster grids, sorting, clipping to study area and masking cloud coverage.

## Resources and references

- GeeksforGeeks. (2026, January 13). Understanding python pickling with example. https://www.geeksforgeeks.org/python/understanding-python-pickling-example/ 
- The Python Software Foundation. (2026, March 20). Glob - unix style pathname pattern expansion. Python documentation. https://docs.python.org/3/library/glob.html
- Project Pythia Notebook template